# IS6400 Kickstarter 数据集探索

本笔记本分析 `Kickstarter Full Campaigns_694062_25_April_2026.csv`。每一行代表一个 Kickstarter 众筹项目。

主要内容：

1. 数据集的 shape、dtypes 和 missing values。
2. 每一列的 attribute type：nominal / ordinal / interval / ratio，以及 discrete / continuous。
3. 选择 `category_name`、`location_name`、`currency` 三个变量，用 summary statistics 和 plots 探索它们之间的关系。
4. 150–200 词总结部分留给你自己填写。

本笔记本应该放在 `group project` 文件夹中运行。

## 导入需要的 Python 包

`pandas` 用来读取和处理表格数据；`matplotlib.pyplot` 用来画图；`numpy` 用来做一些数值计算。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 读取数据

这里把 CSV 文件读成一个 DataFrame。`low_memory=False` 的作用是让 pandas 一次性判断每一列的数据类型，避免因为文件太大而出现混合类型警告。

In [ ]:
CSV_PATH = Path("kickstarter") / "Kickstarter Full Campaigns_694062_25_April_2026.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"找不到数据文件：{CSV_PATH.resolve()}\n"
        "请确认本笔记本放在 group project 文件夹中，或者修改 CSV_PATH。"
    )

df = pd.read_csv(CSV_PATH, low_memory=False)

print(f"数据文件：{CSV_PATH.resolve()}")
print(f"内存占用（大约）：{df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 1. Shape、Dtypes 和 Missing Values

先报告数据集的行列数量，也就是 shape；再检查每一列的 pandas dtype；最后统计每一列的缺失值数量和缺失率。

这里要特别注意：`category_id`、`creator_id`、`location_id` 这类 ID 虽然 pandas 可能读成数字，但它们的含义是“标识符”，不是真正的数量。

In [ ]:
print(f"Shape：{df.shape[0]} 行 × {df.shape[1]} 列")
print("\n各类型列的数量：")
print(df.dtypes.astype(str).value_counts())

In [ ]:
# 对每一列统计：
# dtype：pandas 读取到的数据类型
# missing_count：缺失值数量
# missing_pct：缺失值百分比

missing_values = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})

missing_values = missing_values.sort_values(["missing_count", "missing_pct"], ascending=False)
missing_values.index.name = "column"
missing_values.round({"missing_pct": 2})

## 2. Attribute Type Table

下表给每一列标注两个信息：

- **Measurement scale**：nominal、ordinal、interval 或 ratio。
- **Discrete / Continuous**：取值是可数的离散值，还是可以在某个区间内连续变化的数值。

本数据集中没有明显的 ordinal 变量。时间戳标记为 interval；金额、数量、比例等有真实零点的数值变量标记为 ratio。文本、类别、布尔值和 ID 标记为 nominal。

In [ ]:
# 属性类型映射表。
# 注意：ID 在 CSV 中可能是数字，但在分析意义上应该是 nominal，不应该拿去做加减平均。

attribute_types = {
    "collected_at": ("interval", "discrete"),
    "backers_count": ("ratio", "discrete"),
    "blurb": ("nominal", "discrete"),
    "category_id": ("nominal", "discrete"),
    "category_name": ("nominal", "discrete"),
    "category_parent_id": ("nominal", "discrete"),
    "category_parent_name": ("nominal", "discrete"),
    "category_url": ("nominal", "discrete"),
    "converted_pledged_amount": ("ratio", "continuous"),
    "country": ("nominal", "discrete"),
    "created_at": ("interval", "discrete"),
    "creator_id": ("nominal", "discrete"),
    "creator_name": ("nominal", "discrete"),
    "creator_url": ("nominal", "discrete"),
    "currency": ("nominal", "discrete"),
    "deadline": ("interval", "discrete"),
    "fx_rate": ("ratio", "continuous"),
    "goal": ("ratio", "continuous"),
    "id": ("nominal", "discrete"),
    "launched_at": ("interval", "discrete"),
    "location_country": ("nominal", "discrete"),
    "location_id": ("nominal", "discrete"),
    "location_name": ("nominal", "discrete"),
    "location_state": ("nominal", "discrete"),
    "location_type": ("nominal", "discrete"),
    "name": ("nominal", "discrete"),
    "percent_funded": ("ratio", "continuous"),
    "pledged": ("ratio", "continuous"),
    "profile_blurb": ("nominal", "discrete"),
    "profile_id": ("nominal", "discrete"),
    "profile_name": ("nominal", "discrete"),
    "profile_project_id": ("nominal", "discrete"),
    "profile_state": ("nominal", "discrete"),
    "profile_state_changed_at": ("interval", "discrete"),
    "project_url": ("nominal", "discrete"),
    "spotlight": ("nominal", "discrete"),
    "staff_pick": ("nominal", "discrete"),
    "state": ("nominal", "discrete"),
    "state_changed_at": ("interval", "discrete"),
    "static_usd_rate": ("ratio", "continuous"),
    "usd_exchange_rate": ("ratio", "continuous"),
    "usd_pledged": ("ratio", "continuous"),
    "usd_type": ("nominal", "discrete")
}

attribute_type_table = pd.DataFrame({
    "column": df.columns,
    "pandas_dtype": [str(df[column].dtype) for column in df.columns],
    "measurement_scale": [attribute_types[column][0] for column in df.columns],
    "discrete_or_continuous": [attribute_types[column][1] for column in df.columns],
    "missing_count": [int(df[column].isna().sum()) for column in df.columns],
    "missing_pct": [round(df[column].isna().mean() * 100, 2) for column in df.columns]
})

attribute_type_table

## 3. 变量选择：`category_name`、`location_name`、`currency`

这三个变量都是分类变量，分别回答三个业务问题：

- `category_name`：项目属于什么类型。
- `location_name`：项目从哪里发起。
- `currency`：项目处在哪个货币/筹资市场。

分析单位仍然是项目。这里会先查看每个变量的类别数量、缺失值和最常见取值，再用交叉表和图检查三者之间的关系。

In [ ]:
# Focus on three categorical variables: project type, launch location, and funding currency.
# Copy only the needed columns to keep the exploration lightweight.

categorical_vars = ["category_name", "location_name", "currency"]
selected_categorical_data = df[categorical_vars].copy()

print(f"Number of projects: {len(selected_categorical_data):,}")
print(f"Unique category_name values: {selected_categorical_data['category_name'].nunique(dropna=True):,}")
print(f"Unique location_name values: {selected_categorical_data['location_name'].nunique(dropna=True):,}")
print(f"Unique currency values: {selected_categorical_data['currency'].nunique(dropna=True):,}")

### Summary statistics

这里查看三个分类变量的类别数量、缺失情况和最常见取值。`category_name` 缺失较多，所以要单独看缺失率；`currency` 基本没有缺失。

In [ ]:
# Summarize each categorical variable by its number of levels, missing values, and most frequent value.
# Missing values are not filled here because they are useful quality information.

def summarize_categorical_variable(series, variable_name):
    values = series.dropna()
    value_counts = values.value_counts()
    most_frequent_value = value_counts.index[0]
    most_frequent_count = value_counts.iloc[0]
    return {
        "variable": variable_name,
        "count": values.count(),
        "unique_values": values.nunique(),
        "missing_count": series.isna().sum(),
        "missing_pct": series.isna().mean() * 100,
        "most_frequent_value": most_frequent_value,
        "most_frequent_count": most_frequent_count,
        "most_frequent_pct": most_frequent_count / len(values) * 100
    }

summary_statistics = pd.DataFrame([
    summarize_categorical_variable(selected_categorical_data["category_name"], "category_name"),
    summarize_categorical_variable(selected_categorical_data["location_name"], "location_name"),
    summarize_categorical_variable(selected_categorical_data["currency"], "currency")
]).set_index("variable")

summary_statistics.round(2)

### Relationship statistics

分类变量不适合用 Spearman correlation，所以这里直接统计三者同时出现的最高频组合。为了比较组合占比，先去掉任一变量缺失的行。

In [ ]:
# Crosstabs and grouped counts are more appropriate than correlations for categorical variables.
# Drop rows with any selected-variable missing so the combination percentages are comparable.

complete_cases = selected_categorical_data.dropna(subset=categorical_vars)

top_variable_combinations = (
    complete_cases
    .groupby(categorical_vars)
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("project_count")
    .reset_index()
)
top_variable_combinations["percentage_of_complete_cases"] = (
    top_variable_combinations["project_count"] / len(complete_cases) * 100
)

top_variable_combinations.round({"percentage_of_complete_cases": 2})

### Plot 1：`category_name` 和 `currency` 的关系

先选出最常见的 10 个项目类别和 5 种货币，再计算每个项目类别中各货币的占比。这样可以看出不同类型的项目是否集中在不同货币市场里。

In [ ]:
# Select the most frequent categories and currencies to avoid a chart with too many sparse groups.
top_categories = selected_categorical_data["category_name"].value_counts().head(10).index
top_currencies = selected_categorical_data["currency"].value_counts().head(5).index

category_currency_counts = pd.crosstab(
    selected_categorical_data["category_name"],
    selected_categorical_data["currency"]
).loc[top_categories, top_currencies]
category_currency_share = category_currency_counts.div(
    category_currency_counts.sum(axis=1), axis=0
) * 100

plt.figure(figsize=(10, 6))
fig, ax = plt.subplots(figsize=(11, 6))
category_currency_share.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    colormap="tab20",
    width=0.85
)
ax.set_xlabel("Percentage of projects within category (%)")
ax.set_ylabel("Project category")
ax.set_title("Currency mix for the ten most common project categories")
ax.legend(title="Currency", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Plot 2：`location_name` 和 `category_name` 的关系

先选出最常见的 10 个地点和 10 个类别，再计算每个地点中各类别的占比。热力图颜色越深，表示该地点在这个类别上的集中度越高。

In [ ]:
# Row-normalize the crosstab so locations of different sizes can still be compared.
top_locations = selected_categorical_data["location_name"].value_counts().head(10).index
location_category_data = selected_categorical_data.dropna(subset=["location_name", "category_name"])
location_category_counts = pd.crosstab(
    location_category_data["location_name"],
    location_category_data["category_name"]
).loc[top_locations, top_categories]
location_category_share = location_category_counts.div(
    location_category_counts.sum(axis=1), axis=0
) * 100

fig, ax = plt.subplots(figsize=(12, 7))
heatmap = ax.imshow(location_category_share.values, cmap="viridis", aspect="auto")

ax.set_xticks(np.arange(len(location_category_share.columns)))
ax.set_xticklabels(location_category_share.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(location_category_share.index)))
ax.set_yticklabels(location_category_share.index)

for row_index in range(location_category_share.shape[0]):
    for column_index in range(location_category_share.shape[1]):
        value = location_category_share.iloc[row_index, column_index]
        ax.text(
            column_index,
            row_index,
            f"{value:.1f}%",
            ha="center",
            va="center",
            color="white" if value < location_category_share.values.max() / 2 else "black"
        )

ax.set_xlabel("Project category")
ax.set_ylabel("Launch location")
ax.set_title("Category mix for the ten most common locations")
fig.colorbar(heatmap, ax=ax, label="Percentage of projects within location (%)")
plt.tight_layout()
plt.show()

location_category_share.round(1)

## 4. 150–200 Word Paragraph

请在这里根据运行出的 summary statistics 和两张图，自己写下 150–200 词的发现，以及这些发现对后续建模的意义。